In [1]:
import pathlib

import yaml
import pandas as pd

import jax
import jax.numpy as jnp
import jax.random as jr

import torch

import altair as alt

import trapallada.data
import trapallada.model.signal
import trapallada.model.image
import trapallada.util
import trapallada.core

import ipywidgets as widgets
from sklearn.manifold import TSNE

# TODO

- normalize the signals so that they are between 0 and 1, and then `tanh` en el codificador

So that `impor`ed Python modules are automatically reloaded when modified

In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


The directory containing the data

In [3]:
input_directory = pathlib.Path.cwd().parent / 'data'
assert input_directory.exists()

The particular dataset (specified by a *base* file name)

In [4]:
# base_filename = 'small'
base_filename = '4_classes_modulated'

signals_file = input_directory / f'{base_filename}_signals.pt'
assert signals_file.exists()

metadata_file = input_directory / f'{base_filename}_labels.parquet'
assert metadata_file.exists

Dataset is loaded

In [5]:
# ds = trapallada.data.SignalsDataset(signals_file, metadata_file)
ds = trapallada.data.SignalsDataset(signals_file, metadata_file, normalize=True)
len(ds)

12983

*Base* file name for the results (trained model)

In [6]:
# input_files_stem = pathlib.Path('/tmp') / 'z10_200e'
# input_files_stem = pathlib.Path.cwd().parent /'experiments' / 'z10_200e'
# input_files_stem = pathlib.Path.cwd().parent /'experiments' / 'z10_500e'
# input_files_stem = pathlib.Path.cwd().parent /'experiments' / 'z20_300e'
# input_files_stem = pathlib.Path.cwd().parent /'experiments' / 'new' / 'z_10_300_epochs' 
input_files_stem = pathlib.Path.cwd().parent /'experiments' / 'new' / 'z_50_300_epochs' 
# input_files_stem = pathlib.Path.cwd().parent /'experiments' / 'new' / 'z_2_300_epochs' 
# input_files_stem = pathlib.Path.cwd().parent /'experiments' / 'new' / 'z_3_300_epochs' 


In [7]:
# w_file_upload = widgets.FileUpload(
#     accept='.eqx',
#     multiple=False
# )
# display(w_file_upload)

In [8]:
# w_file_upload.value

From the above *stem*, several file names are derived

In [9]:
# parameters_file = pathlib.Path('parameters.yaml')
parameters_file = input_files_stem.with_suffix('.yaml')
assert parameters_file.exists()

# weights_file = pathlib.Path('foo.eqx')
weights_file = input_files_stem.with_suffix('.eqx')
assert weights_file.exists()

results_file = input_files_stem.with_suffix('.parquet')
assert results_file.exists()

In [10]:
with parameters_file.open() as f:
        parameters = yaml.safe_load(f)
parameters

{'PRNG seeds': {'model initialization': {'decoder': 43, 'encoder': 42},
  'sampling': {'training set': 14, 'validation set': 15},
  'training dataloader': 43,
  'training-validation splitting': 42},
 'encoder': {'input_size': [1, 10000]},
 'loss': {'x_sd': 0.25},
 'model': {'channels': [12, 12, 12],
  'dim_z': 3,
  'kernel_sizes': [128, 64, 32],
  'stride': 2},
 'training': {'batch_size': 32,
  'dl_rng_key': 43,
  'learning_rate': 3e-05,
  'n_epochs': 300,
  'train_test_split': {'test_size': 0.2}}}

In [11]:
results_df = pd.read_parquet(results_file)
results_df

training                             validation                \
             loss loglikelihood         kl          loss loglikelihood   
0    76277.523389 -76264.097644  13.425824  72327.239329 -72313.938357   
1    69452.506034 -69436.858630  15.647252  66908.454125 -66891.130240   
2    65205.757957 -65188.801178  16.956872  63245.022866 -63228.896913   
3    62164.010889 -62147.290938  16.719950  60724.583270 -60707.780678   
4    59956.710144 -59939.912272  16.797872  58869.711938 -58852.653106   
..            ...           ...        ...           ...           ...   
295  36273.423468 -36251.657632  21.765750  55749.852087 -55727.730183   
296  36231.853149 -36209.880451  21.972645  55874.042302 -55852.222752   
297  36156.615198 -36134.745228  21.869911  56026.138005 -56004.800305   
298  36147.821160 -36126.009796  21.811472  55845.654821 -55823.822933   
299  36102.234471 -36080.462314  21.772160  55792.035442 -55770.683213   

                
            kl  
0    13.300724  
1    17.324056  
2    16.125638  
3    16.802479  
4    17.059024  
..         ...  
295  22.122008  
296  21.819555  
297  21.337586  
298  21.831718  
299  21.352243  

[300 rows x 6 columns]

In [12]:
def metric_chart(df: pd.DataFrame, metric: str):

    sub_df = df.loc[:,[('training', metric), ('validation', metric)]]
    sub_df.columns = sub_df.columns.get_level_values(0)
    chart = alt.Chart(
        sub_df.reset_index()
    ).transform_fold(
        sub_df.columns.to_list(), ['dataset', metric]
    ).mark_line().encode(
        x='index:Q', y=f'{metric}:Q', color='dataset:N'
    ).properties(width=600, height=400)

    return chart

metric_chart(results_df, 'loss') | metric_chart(results_df, 'loglikelihood') | metric_chart(results_df, 'kl')

alt.HConcatChart(...)

In [13]:
model = trapallada.model.signal.load_vae(weights_file, parameters_file)

Number of parameters

In [14]:
sum(trapallada.util.get_layers_n_parameters(model).values())

557971

Data loaders for both the training and validation sets

In [15]:
train_dl, validation_dl = trapallada.data.dataloaders_from_dataset(
    ds,
    parameters['training']['train_test_split'] | dict(random_state=parameters['PRNG seeds']['training-validation splitting']),
    parameters['training']['batch_size'], parameters['PRNG seeds']['training dataloader'])

A batch from the **validation** set

In [16]:
# batch = next(iter(train_dl))
batch = next(iter(validation_dl))

In [17]:
# ds.metadata_df

An element from the above batch is encoded, decoded, and a prediction plotted

In [18]:
# i = 0
# i = 1
i = 25

i_s, x_s, y_s = batch
x_s = x_s.numpy()
x_pred, z_mean, z_sd = model(x_s[i], jr.PRNGKey(42))

display(ds.metadata_df.loc[i_s[i].item()])

# first dimension of `x_pred` is 1, and hence it's squeezed out
df = pd.DataFrame({'prediction': x_pred[0], 'actual': x_s[i].ravel()})
n_rows = 2000
alt.Chart(
    df.iloc[:n_rows].reset_index()
).transform_fold(
    df.columns.to_list()
).mark_line().encode(
    x='index:Q', y='value:Q', color='key:N'
).properties(width=1_200, height=400)

label                                 FM
SNR                                3.162
color                               None
file name    FM/FM_SNR3.162_Simul664.mat
Name: 2552, dtype: object

alt.Chart(...)

In [19]:
z_mean, z_sd

(Array([-0.8185089 , -0.16228926,  0.38556987], dtype=float32),
 Array([0.00162329, 0.00125019, 0.00127373], dtype=float32))

In [20]:
df = trapallada.util.make_projections_df(ds, validation_dl, model.encoder)
# df = trapallada.util.make_projections_df(ds, train_dl, model.encoder)
df.head()

,label,SNR,color,file name,z_1,z_2,z_3,tsne_1,tsne_2
8940,carrier,3.162,None,singlePeriod/carrier_SNR3.162_Simul931.mat,-0.510910,-1.049176,0.978471,3.684055,-7.225535
7298,AM,31.620,None,AM/AM_SNR31.62_Simul641.mat,2.491498,-0.653564,1.984414,-23.958952,38.072338
10764,carrier,3.162,None,singlePeriod/carrier_SNR3.162_Simul853.mat,0.348502,-0.705042,1.341208,-18.801510,1.602071
1331,FM,31.620,None,FM/FM_SNR31.62_Simul524.mat,-0.311241,-2.455626,1.325775,12.794421,19.111103
10608,carrier,3.162,None,singlePeriod/carrier_SNR3.162_Simul60.mat,-2.868409,0.799369,0.467378,9.242909,-51.691254


Signals that are just noise have no SNR, but that's fine for...

In [21]:
df['SNR'].isna().sum()

197

...plotting

In [22]:
alt.Chart(df.sample(n=1_000)).mark_point().encode(
    x='tsne_1:Q',
    y='tsne_2:Q',
    color='label:N',
    fill='label:N',
    size='SNR:Q',
    tooltip=['file name:N']
).properties(
    width=800, height=600
).interactive()

/home/manu/mambaforge/envs/trap/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)


alt.Chart(...)

The latent space is projected into a 3D space

In [23]:
df_3 = trapallada.util.make_projections_df(ds, validation_dl, model.encoder, n_projected_components=3)
df_3.head()

,label,SNR,color,file name,z_1,z_2,z_3,tsne_1,tsne_2,tsne_3
8940,carrier,3.162,None,singlePeriod/carrier_SNR3.162_Simul931.mat,-0.510910,-1.049176,0.978471,3.636283,-0.461162,-1.262445
7298,AM,31.620,None,AM/AM_SNR31.62_Simul641.mat,2.491498,-0.653564,1.984414,-8.866012,5.760436,-0.349990
10764,carrier,3.162,None,singlePeriod/carrier_SNR3.162_Simul853.mat,0.348502,-0.705042,1.341208,-3.246165,1.909572,-1.735243
1331,FM,31.620,None,FM/FM_SNR31.62_Simul524.mat,-0.311241,-2.455626,1.325775,7.248997,7.123806,3.356147
10608,carrier,3.162,None,singlePeriod/carrier_SNR3.162_Simul60.mat,-2.868409,0.799369,0.467378,3.457542,-12.072311,3.474161


In [24]:
import plotly.express as px
fig = px.scatter_3d(df_3, x='tsne_1', y='tsne_2', z='tsne_3', color='label', opacity=0.5, width=800, height=800)
fig.show()